In [ ]:
import ssl
from cognite.client import CogniteClient, ClientConfig, global_config
from cognite.client.credentials import OAuthClientCredentials

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd

# Resolve project root: one level up from this notebook's directory
_nb_dir = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_nb_dir, ".."))

In [ ]:
load_dotenv()

In [ ]:
ctx = ssl.create_default_context()
ctx.load_verify_locations(os.path.join(PROJECT_ROOT, "combined-ca-bundle.pem"))
global_config.ssl_context = ctx

In [ ]:
# This value will depend on the cluster your CDF project runs on
cluster = os.environ["OID_COGNITE_CDF_CLUSTER"]
base_url = f"https://{cluster}.cognitedata.com"
tenant_id = os.environ["OID_COGNITE_TENANT_ID"]
client_id = os.environ["OID_COGNITE_CLIENT_ID"]
client_secret = os.environ["OID_COGNITE_CLIENT_SECRET"]


In [ ]:
creds = OAuthClientCredentials(
  token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
  client_id=client_id,
  client_secret=client_secret,
  scopes=[f"{base_url}/.default"]
)

cnf = ClientConfig(
  client_name=client_id,
  base_url=base_url,
  project="publicdata",
  credentials=creds
)

global_config.default_client_config = cnf
c = CogniteClient()

In [ ]:
COMPRESSOR_ASSETS = [
    "23-KA-9101",   # 1st-stage compressor
    "23-HA-9103",   # suction cooler
    "23-HA-9114",   # discharge cooler A
    "23-HA-9115",   # discharge cooler B
    "23-VG-9101",   # suction scrubber
    "23-FV-92543",  # anti-surge valve
    "23-PV-92538",  # suction throttle valve
]

In [ ]:
rows = []
for asset_name in COMPRESSOR_ASSETS:
    assets = c.assets.search(name=asset_name, limit=5)
    if not assets:
        print(f"⚠ No asset found for '{asset_name}'")
        continue
    asset = assets[0]
    ts_list = c.time_series.list(asset_subtree_ids=[asset.id], limit=None)
    print(f"{asset.name} (id={asset.id}): {len(ts_list)} time series")
    for ts in ts_list:
        rows.append({
            "external_id": ts.external_id,
            "name": ts.name,
            "description": ts.description,
            "unit": ts.unit,
            "asset_id": ts.asset_id,
            "asset_name": asset.name,
        })

df = pd.DataFrame(rows)
print(f"\nTotal: {len(df)} candidate tags")
output_path = os.path.join(PROJECT_ROOT, "data", "raw", "tag_candidates.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
df

## Quick data exploration — verify key tags have data
Pull 1 week of 10-min averages for the tags that were previously gaps:
discharge pressure, discharge temperature, gas flow, and one vibration probe.
Plus suction pressure and motor current as baseline sanity checks.

In [ ]:
from datetime import datetime

SPOT_CHECK_TAGS = {
    "p_suction":    "VAL_23-PT-92532:X.Value",
    "p_discharge":  "VAL_23_PT_92537:Z.X.Value",
    "t_discharge":  "VAL_23_TT_92536:Z.X.Value",
    "flow_gas":     "VAL_23_FT_92537_03:Z.X.Value",
    "vib_mtr_de_1": "VAL_23_YT_96134_01:Z.X.Value",
    "motor_current":"VAL_23-KA-9101-M01_Current_L1:VALUE",
}

# Pull 1 week of 10-min averages
start = datetime(2024, 3, 1)
end   = datetime(2024, 3, 8)

dfs = {}
for label, ext_id in SPOT_CHECK_TAGS.items():
    try:
        result = c.time_series.data.retrieve(
            external_id=ext_id,
            aggregates=["average"],
            granularity="10m",
            start=start,
            end=end,
        )
        if result is not None and len(result) > 0:
            df_tag = result.to_pandas()
            df_tag.columns = [label]
            dfs[label] = df_tag
            pct_null = df_tag[label].isna().mean() * 100
            print(f"✓ {label:18s} | {len(df_tag):5d} rows | {pct_null:.1f}% null | "
                  f"range: {df_tag[label].min():.2f} – {df_tag[label].max():.2f}")
        else:
            print(f"✗ {label:18s} | NO DATA returned")
    except Exception as e:
        print(f"✗ {label:18s} | ERROR: {e}")

In [ ]:
import plotly.subplots as sp
import plotly.graph_objects as go

if dfs:
    n = len(dfs)
    fig = sp.make_subplots(rows=n, cols=1, shared_xaxes=True,
                           subplot_titles=list(dfs.keys()),
                           vertical_spacing=0.04)
    for i, (label, df_tag) in enumerate(dfs.items(), 1):
        fig.add_trace(
            go.Scatter(x=df_tag.index, y=df_tag[label], mode="lines", name=label),
            row=i, col=1,
        )
    fig.update_layout(height=250 * n, showlegend=False,
                      title_text="1-week spot check (2024-03-01 to 2024-03-08)")
    fig.show()
else:
    print("No data to plot")

In [ ]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = "${api-key}"

client = boto3.client("bedrock-runtime", region_name="us-east-1")

response = client.converse(
    modelId="us.anthropic.claude-sonnet-4-6",
    messages=[{"role": "user", "content": [{"text": "Hello"}]}]
)

In [ ]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

br = boto3.client('bedrock-runtime', region_name='us-east-1')

resp = br.converse(
    modelId='openai.gpt-oss-120b',
    messages=[{'role': 'user', 'content': [{'text': 'Hello, say hi in one sentence.'}]}],
    inferenceConfig={'maxTokens': 100},
)
print(resp['output']['message']['content'][0]['text'])

In [ ]:
from openai import OpenAI
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key='ABS...your-key',
    base_url='https://bedrock-runtime.us-east-2.amazonaws.com/openai/v1',
)
resp = client.chat.completions.create(
    model='your-model-id',
    messages=[{'role': 'user', 'content': 'Hello, say hi in one sentence.'}],
    max_tokens=100,
)
print(resp.choices[0].message.content)

In [ ]:
os.getenv('AWS_BEARER_TOKEN_BEDROCK')

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
import boto3

br = boto3.client('bedrock-runtime', region_name='us-east-1')
resp = br.converse(
    modelId='anthropic.claude-sonnet-4-20250514',
    messages=[{'role': 'user', 'content': [{'text': 'Hello, say hi in one sentence.'}]}],
    inferenceConfig={'maxTokens': 100},
)
print(resp['output']['message']['content'][0]['text'])

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="global.openai.gpt-5.6-sol",
    messages=[{"role": "user", "content": "Can you explain the features of Amazon Bedrock?"}]
    )
print(response)

In [ ]:
import boto3

client = boto3.client('bedrock-runtime', region_name='us-east-1')
response = client.converse(
    modelId='amazon.nova-micro-v1:0',
    messages=[
        {
            'role': 'user',
            'content': [{'text': 'Can you explain the features of Amazon Bedrock?'}]
        }
    ]
)
print(response)


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()

client = OpenAI()

stream = client.responses.create(
    model="openai.gpt-oss-20b",
    input=[
        {"role": "user", "content": "Tell me a short story about a robot."}
    ],
    stream=True
)

for event in stream:
    print(event)